# 工况牌目标搜索工具

本 Notebook 只测试“云台搜索 + 拍照 + StepFun 多模态判断”。小车底盘不会移动。

In [1]:
# 可以留空，脚本会读取环境变量 STEPFUN_API_KEY 或当前目录 stepfun_api_key.txt。
API_KEY = ""

OUTPUT_DIR = "target_search_output"
TARGET_TEXT = "工业工况巡检牌"
MODEL = "step-3.7-flash"

# 已校准的正视位置
HOME_PAN = 2048
HOME_TILT = 2868

# 搜索范围。若目标容易出现在更靠左右两侧，可把 SEARCH_PROFILE 改为 "front_sign_wide"。
SEARCH_PROFILE = "front_sign_wide"
PAN_DELTA = 620
TILT_UP_DELTA = 120
TILT_DOWN_DELTA = 260

# 目标搜索属于语义理解阶段，不压缩输出预算。
MAX_TOKENS = 4096

In [2]:
from target_search_tool import run_demo, summarize_report

report = run_demo(
    output_dir=OUTPUT_DIR,
    target_text=TARGET_TEXT,
    api_key=API_KEY or None,
    model=MODEL,
    reasoning_effort="low",
    home_pan=HOME_PAN,
    home_tilt=HOME_TILT,
    pan_delta=PAN_DELTA,
    tilt_up_delta=TILT_UP_DELTA,
    tilt_down_delta=TILT_DOWN_DELTA,
    search_profile=SEARCH_PROFILE,
    camera_width=640,
    camera_height=480,
    capture_samples=3,
    pre_capture_settle=0.30,
    max_tokens=MAX_TOKENS,
    return_to_home=True,
)
print(summarize_report(report))

serial Open!
serial Close!
Target Search Tool
Summary: Target sign found.
Parse OK: True
Partial parse: False
Target found: True
Best pose: left_wide
Risk level: abnormal
JSON report: target_search_output/target_search_report.json
Text report: target_search_output/target_search_report.txt


In [3]:
# 查看模型结构化判断
report.get("vision", {}).get("parsed", {})

{'target_found': True,
 'best_pose': 'left_wide',
 'confidence': 0.95,
 'visible_text': '设备A01, 86°C, OVERHEAT, DANGER, ELECTRICAL HAZARD, KEEP AWAY, 请识别设备状态与现场风险',
 'pose_results': '该视角清晰展示了工业工况巡检牌全貌，包含设备图片、温度状态、风险警示等关键信息',
 'risk_level': 'abnormal',
 'inspection_summary': '设备A01显示温度86°C，有过热(OVERHEAT)警告，存在电气危险(DANGER/ELECTRICAL HAZARD)，需立即处置',
 'reason': 'left_wide图完整呈现了包含设备图片、状态文字(86°C)、风险提示(OVERHEAT/DANGER)的工业工况巡检牌，符合目标定义；画面中出现明确异常信号(过热、危险标识)，依据风险分级标准判定为abnormal',
 'next_action': '立即上报设备过热及电气危险情况，建议停机检查、采取降温措施，并设置警戒区域禁止靠近'}

In [4]:
# 查看每个视角保存的图片路径
[(item.get("pose"), item.get("path")) for item in report.get("candidate_images", [])]

[('center', 'target_search_output/center_best.jpg'),
 ('left', 'target_search_output/left_best.jpg'),
 ('right', 'target_search_output/right_best.jpg'),
 ('left_wide', 'target_search_output/left_wide_best.jpg'),
 ('right_wide', 'target_search_output/right_wide_best.jpg'),
 ('down', 'target_search_output/down_best.jpg')]